In [ ]:
# imports
import os, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report
import mlflow, mlflow.sklearn

In [ ]:
# load raw dataset
df = pd.read_csv('../data/raw/skill_builder_data.csv', encoding='latin1', low_memory=False)
print(df.shape)
df.head()

In [ ]:
# quick look at columns and null counts
print(df.dtypes)
print(df.isnull().sum())

In [ ]:
# target distribution
print(df['correct'].value_counts())

In [ ]:
# keep only valid binary targets
df = df[df['correct'].isin([0, 1])].copy()
df['correct'] = df['correct'].astype(int)

# baseline features (will be extended after EDA by teammate)
features = ['attempt_count', 'ms_first_response', 'hint_count', 'hint_total', 'overlap_time', 'opportunity']

# impute missing values with column median
for col in features:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

# sample 100k rows for faster prototyping
df = df.sample(n=100000, random_state=42)

X = df[features]
y = df['correct']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

In [ ]:
# connect to DAGsHub MLflow tracking server
os.environ['MLFLOW_TRACKING_USERNAME'] = 'suhaybfr'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '80f88654df59c0d94cfb596dcb617803b50d0ffb'
mlflow.set_tracking_uri('https://dagshub.com/suhaybfr/ML-ops.mlflow')
mlflow.set_experiment('student_performance_baseline')

In [ ]:
# train random forest and log run to mlflow
with mlflow.start_run(run_name='notebook_baseline'):

    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc     = accuracy_score(y_test, y_pred)
    prec    = precision_score(y_test, y_pred, zero_division=0)
    rec     = recall_score(y_test, y_pred, zero_division=0)
    f1      = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)

    # log params + metrics to dagshub
    mlflow.log_params({'n_estimators': 100, 'max_depth': 10})
    mlflow.log_metrics({'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'roc_auc': roc_auc})

    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1:        {f1:.4f}')
    print(f'ROC-AUC:   {roc_auc:.4f}')

In [ ]:
# full classification report
print(classification_report(y_test, y_pred))

In [ ]:
# feature importance ranking
pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)